# Month-end finance analysis
Revenue, DSO trend, collections vs billing, and overdue accounts. Run from this directory. Amounts assume a single reporting currency; timestamps use the warehouse date. Rolling windows include the current partial month.

In [ ]:
import os

from finance_nb.warehouse import connect, run_query

# FINANCE_WAREHOUSE: teradata (default), teradata-emulator (old path, local) or databricks.
# FINANCE_AS_OF pins CURRENT_DATE, e.g. 2026-11-09; unset uses the warehouse date.
target = os.getenv("FINANCE_WAREHOUSE", "teradata")
config = {
    "host": "tdprod-fin.otterworks.internal",
    "user": "nb_dsci_pkhan",
    "password": "not-a-real-password",
    "database": "SFX_BILLING",
}
connection = connect(config, target=target)
print("target:", target, "as-of:", os.getenv("FINANCE_AS_OF") or "warehouse CURRENT_DATE")

## Revenue by month

In [ ]:
revenue_sql = """SEL CAST(CAST(INVOICE_DT AS FORMAT 'YYYY-MM') AS CHAR(7)) AS MONTH_LABEL,
  SUM(TOTAL_AMOUNT - ZEROIFNULL(TAX_AMOUNT)) AS REVENUE
FROM SFX_BILLING.INVOICE_HDR
WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE, -12) AND INVOICE_DT >= 1200101
GROUP BY 1 ORDER BY 1"""
revenue = run_query(connection, revenue_sql, "revenue")
revenue

## DSO trend

In [ ]:
dso_sql = """WITH billed AS (
  SEL CAST(CAST(INVOICE_DT AS FORMAT 'YYYY-MM') AS CHAR(7)) AS MONTH_LABEL,
    SUM(TOTAL_AMOUNT) AS BILLED
  FROM SFX_BILLING.INVOICE_HDR WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE, -12)
    AND INVOICE_DT >= 1200101 GROUP BY 1
), open_ar AS (
  SEL b.MONTH_LABEL, SUM(i.TOTAL_AMOUNT - ZEROIFNULL(p.PAID)) AS OPEN_AR
  FROM billed b JOIN SFX_BILLING.INVOICE_HDR i
    ON i.INVOICE_DT <= LAST_DAY(CAST(b.MONTH_LABEL || '-01' AS DATE FORMAT 'YYYY-MM-DD'))
  LEFT JOIN (
    SEL b.MONTH_LABEL, p.INVOICE_ID, SUM(p.AMOUNT) AS PAID
    FROM billed b JOIN SFX_BILLING.PAYMENT p
      ON p.PAYMENT_DT <= LAST_DAY(CAST(b.MONTH_LABEL || '-01' AS DATE FORMAT 'YYYY-MM-DD'))
    GROUP BY 1, 2
  ) p ON p.INVOICE_ID = i.INVOICE_ID AND p.MONTH_LABEL = b.MONTH_LABEL
  WHERE i.INVOICE_DT >= 1200101 GROUP BY 1
), ratios AS (
  SEL b.MONTH_LABEL, a.OPEN_AR / NULLIFZERO(b.BILLED) * 30 AS DSO
  FROM billed b JOIN open_ar a ON a.MONTH_LABEL = b.MONTH_LABEL
)
SEL MONTH_LABEL, ZEROIFNULL(DSO) AS DSO FROM ratios ORDER BY 1"""
dso = run_query(connection, dso_sql, "dso")
dso

## Collections vs billing

In [ ]:
collections_sql = """WITH billed AS (
  SEL CAST(CAST(INVOICE_DT AS FORMAT 'YYYY-MM') AS CHAR(7)) AS MONTH_LABEL,
    SUM(TOTAL_AMOUNT) AS BILLED
  FROM SFX_BILLING.INVOICE_HDR WHERE INVOICE_DT >= CURRENT_DATE - 90
    AND INVOICE_DT >= 1200101 GROUP BY 1
), cash AS (
  SEL CAST(CAST(PAYMENT_DT AS FORMAT 'YYYY-MM') AS CHAR(7)) AS MONTH_LABEL,
    SUM(AMOUNT) AS COLLECTED
  FROM SFX_BILLING.PAYMENT WHERE PAYMENT_DT >= CURRENT_DATE - 90 GROUP BY 1
)
SEL COALESCE(b.MONTH_LABEL, c.MONTH_LABEL) AS MONTH_LABEL,
  ZEROIFNULL(b.BILLED) AS BILLED, ZEROIFNULL(c.COLLECTED) AS COLLECTED
FROM billed b FULL OUTER JOIN cash c ON c.MONTH_LABEL = b.MONTH_LABEL ORDER BY 1"""
collections = run_query(connection, collections_sql, "collections")
collections

## Top overdue accounts

In [ ]:
overdue_sql = """WITH balances AS (
  SEL i.CUSTOMER_ID, i.INVOICE_DT + 30 AS DUE_DT, i.TOTAL_AMOUNT - ZEROIFNULL(p.PAID) AS OPEN_BALANCE
  FROM SFX_BILLING.INVOICE_HDR i
  LEFT JOIN (SEL INVOICE_ID, SUM(AMOUNT) AS PAID FROM SFX_BILLING.PAYMENT GROUP BY INVOICE_ID) p
    ON p.INVOICE_ID = i.INVOICE_ID
  WHERE i.INVOICE_DT >= 1200101 AND (i.INVOICE_DT + 30) < CURRENT_DATE - 30
)
SEL TOP 10 a.ACCOUNT_NUMBER AS ACCOUNT_NO, a.ACCOUNT_NAME AS CUSTOMER_NAME,
  CAST(MIN(b.DUE_DT) AS DATE FORMAT 'YYYY-MM-DD') AS OLDEST_DUE_DATE,
  SUM(b.OPEN_BALANCE) AS OVERDUE_BALANCE
FROM balances b JOIN SFX_BILLING.CUSTOMER_ACCOUNT a ON a.CUSTOMER_ID = b.CUSTOMER_ID
WHERE b.OPEN_BALANCE > 0 GROUP BY a.ACCOUNT_NUMBER, a.ACCOUNT_NAME
ORDER BY OVERDUE_BALANCE DESC, a.ACCOUNT_NUMBER"""
overdue = run_query(connection, overdue_sql, "overdue")
overdue

In [ ]:
connection.close()